# LLM API Demo

This notebook demonstrates using LLMs via OpenAI-compatible APIs from JupyterHub.

Supports **OpenAI** (default) or **Ollama Cloud** as the provider.

In [ ]:
%pip install -q openai

In [ ]:
from openai import OpenAI

# === PICK YOUR PROVIDER ===
# Option 1: OpenAI (fast, reliable)
PROVIDER = "openai"
API_KEY = "your-openai-key-here"
BASE_URL = "https://api.openai.com/v1"
MODEL = "gpt-4.1-nano"

# Option 2: Ollama Cloud (free, but slow — may time out on free tier)
# PROVIDER = "ollama"
# API_KEY = "your-ollama-key-here"
# BASE_URL = "https://ollama.com/v1"
# MODEL = "qwen3.5:cloud"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=120)
print(f"Provider: {PROVIDER} | Model: {MODEL}")

## 1. Simple Chat Completion

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "Explain Kubernetes in 3 sentences."},
    ],
)

print(response.choices[0].message.content)

## 2. Streaming Response

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "List 5 benefits of GitOps for Kubernetes."},
    ],
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
print()

## 3. Tool Calling (Function Calling)

In [ ]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_cluster_status",
            "description": "Get the status of a Kubernetes cluster by name",
            "parameters": {
                "type": "object",
                "properties": {
                    "cluster_name": {
                        "type": "string",
                        "description": "The name of the cluster, e.g. team-ml-dev",
                    }
                },
                "required": ["cluster_name"],
            },
        },
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the status of the team-ml-dev cluster?"},
    ],
    tools=tools,
    tool_choice="auto",
)

msg = response.choices[0].message
if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f"Tool: {tc.function.name}")
        print(f"Args: {tc.function.arguments}")
else:
    print(msg.content)

## 4. Multi-turn Conversation

In [ ]:
messages = [
    {"role": "system", "content": "You are a Kubernetes expert. Be concise."},
    {"role": "user", "content": "What is a vcluster?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages)
assistant_msg = response.choices[0].message.content
print(f"Assistant: {assistant_msg}\n")

messages.append({"role": "assistant", "content": assistant_msg})
messages.append({"role": "user", "content": "How is it different from a namespace?"})

response = client.chat.completions.create(model=MODEL, messages=messages)
print(f"Assistant: {response.choices[0].message.content}")